In [4]:
import pandas as pd
import requests
import os
# Questo serve per vedere meglio le tabelle nel notebook
from IPython.display import display 

os.makedirs("../data/files", exist_ok=True)

In [5]:
# Carichiamo il campione che hai scaricato prima
file_parquet = "../data/raw/documenti_sample_5000.parquet"
df = pd.read_parquet(file_parquet)

# Visualizziamo i dati in modo elegante
display(df.head(10))

,id,source,release_batch,volume,original_filename,page_count,size,document_description,source_url,path,has_thumbnail,ui_document_folder_path,tags,duration_seconds,is_ready
0,COURT_bryant-v-indyke-119cv10479_001.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,001.pdf,16,516800,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
1,COURT_bryant-v-indyke-119cv10479_002.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,002.pdf,2,478293,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
2,COURT_bryant-v-indyke-119cv10479_003.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,003.pdf,2,220533,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
3,COURT_bryant-v-indyke-119cv10479_004.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,004.pdf,2,308451,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
4,COURT_bryant-v-indyke-119cv10479_005.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,005.pdf,1,144541,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
5,COURT_bryant-v-indyke-119cv10479_006.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,006.pdf,5,122055,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
6,COURT_bryant-v-indyke-119cv10479_007.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,007.pdf,1,578403,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
7,COURT_bryant-v-indyke-119cv10479_008.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,008.pdf,1,72515,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
8,COURT_bryant-v-indyke-119cv10479_009.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,009.pdf,2,39217,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True
9,COURT_bryant-v-indyke-119cv10479_010.pdf,doj-court-records,"Bryant v. Indyke, No. 119-cv-10479 (S.D.N.Y. 2...",<NA>,010.pdf,2,225890,NaN,https://www.justice.gov/multimedia/Court%20Rec...,"court-records/Bryant v. Indyke, No. 119-cv-104...",True,"court-records/Bryant v. Indyke, No. 119-cv-104...","{""year"":""2019"",""court"":""S.D.N.Y"",""people"":[""ju...",<NA>,True


In [ ]:
def download_document(row):
    # Pulizia ID: rimuoviamo estensioni fastidiose (.pdf, .parquet, ecc)
    doc_id_clean = str(row['id']).split('.')[0]
    nome_file = row['original_filename']
    
    # Proviamo prima jdrive, poi files
    endpoints = ["jdrive", "files"]
    
    for ep in endpoints:
        url = f"https://data.jmail.world/v1/{ep}/{doc_id_clean}"
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(f"data/raw/{nome_file}", "wb") as f:
                    f.write(r.content)
                return f"Successo ({ep})"
        except:
            continue
    return "Fallito"

# Testiamo il download sui primi 5 file
df_test = df.head(5).copy()
df_test['status'] = df_test.apply(download_document, axis=1)
display(df_test[['id', 'original_filename', 'status']])

,id,original_filename,status
0,COURT_bryant-v-indyke-119cv10479_001.pdf,001.pdf,Fallito
1,COURT_bryant-v-indyke-119cv10479_002.pdf,002.pdf,Fallito
2,COURT_bryant-v-indyke-119cv10479_003.pdf,003.pdf,Fallito
3,COURT_bryant-v-indyke-119cv10479_004.pdf,004.pdf,Fallito
4,COURT_bryant-v-indyke-119cv10479_005.pdf,005.pdf,Fallito
